In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
from src.data_prep import load_and_clean, FEATURES, TARGET
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

df = load_and_clean()
X, y = df[FEATURES], np.log1p(df[TARGET])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat, num = ["Neighborhood"], ["GrLivArea", "BedroomAbvGr", "Bathrooms", "GarageCars"]
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                         ("num", "passthrough", num)])

In [2]:
def r2_dollar(y_true, y_pred):
    from sklearn.metrics import r2_score
    return r2_score(np.expm1(y_true), np.expm1(y_pred))

for name, m in {"Linear": LinearRegression(),
                "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
                "GradBoost": GradientBoostingRegressor(random_state=42)}.items():
    pipe = Pipeline([("pre", pre), ("model", m)]).fit(X_train, y_train)
    print(f"{name:14s} train R2={r2_dollar(y_train, pipe.predict(X_train)):.3f}"
          f"   test R2={r2_dollar(y_test, pipe.predict(X_test)):.3f}")

Linear         train R2=0.841   test R2=0.781
RandomForest   train R2=0.975   test R2=0.799
GradBoost      train R2=0.886   test R2=0.795


In [3]:
from sklearn.metrics import make_scorer, r2_score, mean_absolute_error, mean_squared_error

r2_scorer   = make_scorer(lambda t, p: r2_score(np.expm1(t), np.expm1(p)))
mae_scorer  = make_scorer(lambda t, p: mean_absolute_error(np.expm1(t), np.expm1(p)),
                          greater_is_better=False)

In [4]:
from sklearn.model_selection import cross_val_score
rf_pipe = Pipeline([("pre", pre), ("model", RandomForestRegressor(n_estimators=200, random_state=42))])
scores = cross_val_score(rf_pipe, X, y, cv=5, scoring=mae_scorer)
print("MAE 5-fold: $%.0f ± $%.0f" % (-scores.mean(), scores.std()))

MAE 5-fold: $23155 ± $982


In [5]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "model__n_estimators":     [200, 400],
    "model__max_depth":        [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features":     ["sqrt", 0.5],
}
grid = GridSearchCV(rf_pipe, param_grid, cv=5, scoring=mae_scorer, n_jobs=-1)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV MAE: $%.0f" % -grid.best_score_)

Best params: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__n_estimators': 400}
Best CV MAE: $22737


In [6]:
best = grid.best_estimator_
pred = np.expm1(best.predict(X_test)); actual = np.expm1(y_test)
print("MAE $%.0f | RMSE $%.0f | R2 %.3f" % (
    mean_absolute_error(actual, pred),
    np.sqrt(mean_squared_error(actual, pred)),
    r2_score(actual, pred)))

MAE $22826 | RMSE $32674 | R2 0.807
